# Verdict Workstation Gemma Bake-Off
Run this notebook on a free GPU runtime. It trains only on the provider-neutral JSONL exports and never uploads portfolio data. The 30-case evaluation set is deliberately absent from the pilot.

In [ ]:
!pip install -q --upgrade unsloth datasets trl transformers accelerate bitsandbytes

In [ ]:
!git clone -q https://github.com/jainmrigank/Verdict-Workstation.git
%cd '/content/Verdict-Workstation/New project/stock-analysis-app'

Upload `pilot_train.jsonl` and `pilot_validation.jsonl`, produced locally by `python3 tools/llm_dataset.py export --pilot`.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
target = Path('data/training/provider')
target.mkdir(parents=True, exist_ok=True)
for name, payload in uploaded.items():
    if name not in {'pilot_train.jsonl', 'pilot_validation.jsonl'}:
        raise ValueError(f'Unexpected file: {name}')
    (target / name).write_bytes(payload)

In [ ]:
!python3 tools/gemma_bakeoff.py --dry-run --model gemma4-e4b

In [ ]:
!python3 tools/gemma_bakeoff.py --model gemma4-e4b --epochs 2 --lora-rank 16 --export-gguf

If the 12B candidate fits the assigned accelerator without truncation, repeat the dry run and training command with `--model gemma4-12b`. Download the resulting adapter, GGUF, and `run_manifest.json`; model artifacts remain untracked.